# Câu 3b. Xây dựng vector đặc trưng văn bản: N-grams (Bag-of-N-grams)
**Input:** `../Data_Preprocessing` (kết quả Câu 1), mỗi file là 1 văn bản, mỗi dòng là 1 câu
**Output:** ma trận Bag-of-N-grams (số văn bản × số n-gram), lưu file `NGRAM.csv`
*(Tham khảo Ví dụ 4.3a, 4.3c – Chương 4)*

N-gram được tạo **trong từng câu** (mỗi dòng của file), để không ghép nhầm từ cuối câu này với từ đầu câu sau. Đổi `n` để dùng trigram (n=3).

In [ ]:
# pip install scikit-learn pandas
import os
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

n = 2   # bigram

## Đọc dữ liệu (giữ nguyên từng câu)

In [ ]:
path_in = "../Data_Preprocessing"

List_Files = sorted([f for f in os.listdir(path_in) if f.endswith(".txt")])
D = []          # mỗi phần tử là danh sách các câu của 1 văn bản
Doc_Names = []
for file_name in List_Files:
    f = open(os.path.join(path_in, file_name), "r", encoding="utf-8")
    List_Sentences = [s for s in f.read().split("\n") if s.strip() != ""]
    f.close()
    if len(List_Sentences) > 0:
        D.append(List_Sentences)
        Doc_Names.append(file_name)

print("Số văn bản:", len(D))
print("Các câu của văn bản đầu tiên (" + Doc_Names[0] + "):", D[0])

## Cách 1: Python thuần (Ví dụ 4.3a)

In [ ]:
def generate_ngrams(text, n):
    # Tách từ
    words = text.split()
    # Sử dụng hàm zip để tách các cặp từ liên tiếp trong danh sách các từ
    ngrams = zip(*[words[i:] for i in range(n)])
    # Sử dụng list comprehension để tạo n-grams
    return [" ".join(ngram) for ngram in ngrams]

print(generate_ngrams(D[0][0], n))

In [ ]:
# Bước 1: Xây dựng n-grams cho từng văn bản
texts = []
for List_Sentences in D:
    ngrams_doc = []
    for sentence in List_Sentences:
        ngrams_doc = ngrams_doc + generate_ngrams(sentence, n)
    texts.append(ngrams_doc)

# Bước 2: Xây dựng tập từ vựng n-grams (sắp xếp theo alphabet)
dictionary = sorted(set(g for doc in texts for g in doc))
print("Số n-gram trong từ điển:", len(dictionary))
print("20 n-gram đầu tiên:", dictionary[:20])

In [ ]:
# Bước 3: Tạo vector đặc trưng (đếm số lần mỗi n-gram xuất hiện trong văn bản)
def bag_of_ngram(ngrams_doc):
    wordDict = dict.fromkeys(dictionary, 0)
    for g in ngrams_doc:
        wordDict[g] += 1
    return list(wordDict.values())

NGRAM_1 = [bag_of_ngram(doc) for doc in texts]
df_ngram_1 = pd.DataFrame(NGRAM_1, columns=dictionary, index=Doc_Names)
print("Kích thước ma trận Bag-of-N-grams:", df_ngram_1.shape)
df_ngram_1.head()

## Cách 2: Sử dụng thư viện sklearn (Ví dụ 4.3c)
Đưa từng câu vào `CountVectorizer(ngram_range=(n, n))`, sau đó cộng các câu thuộc cùng 1 văn bản lại.

In [ ]:
Data_Sentences = []   # tất cả các câu
Sentence_Doc = []     # câu thuộc văn bản nào
for i in range(len(D)):
    for sentence in D[i]:
        Data_Sentences.append(sentence)
        Sentence_Doc.append(Doc_Names[i])

vectorizer = CountVectorizer(ngram_range=(n, n), token_pattern=r"(?u)\S+", lowercase=False)
features = vectorizer.fit_transform(Data_Sentences)

df_sent = pd.DataFrame(features.toarray(), columns=vectorizer.get_feature_names_out())
df_ngram_2 = df_sent.groupby(Sentence_Doc).sum().loc[Doc_Names]
print("Kích thước ma trận Bag-of-N-grams:", df_ngram_2.shape)
df_ngram_2.head()

## So sánh 2 cách và lưu kết quả

In [ ]:
print("Hai cách cho kết quả giống nhau:", (df_ngram_1.values == df_ngram_2[dictionary].values).all())

# Top 20 n-gram xuất hiện nhiều nhất trong toàn bộ bộ dữ liệu
print(df_ngram_1.sum().sort_values(ascending=False).head(20))

df_ngram_1.to_csv("NGRAM.csv", encoding="utf-8-sig")
print("Đã lưu NGRAM.csv")